# A CNN in NumPy

One convolutional layer, hand-written forward and backward passes, no deep learning framework.

`28x28` -> conv (2 filters, 3x3, sigmoid) -> `2x26x26` -> average pool 2x2 -> `2x13x13` -> flatten `338` -> dense -> `10` logits

In [ ]:
import os
import urllib.request

import numpy as np

In [ ]:
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"


def load_data(path="mnist.npz"):
    """Fetch MNIST once, cache it next to the notebook, return it scaled to [0, 1]."""
    if not os.path.exists(path):
        print(f"Downloading data from {MNIST_URL}")
        urllib.request.urlretrieve(MNIST_URL, path)
    with np.load(path) as data:
        x_train, y_train = data["x_train"], data["y_train"]
        x_test, y_test = data["x_test"], data["y_test"]
    return x_train / 255.0, y_train, x_test / 255.0, y_test


def one_hot_encode(labels, num_classes=10):
    return np.eye(num_classes)[labels]

In [ ]:
def sigmoid(x):
    x = np.clip(x, -500, 500)  # clamp before exp so large negatives do not overflow
    return 1 / (1 + np.exp(-x))


def sigmoid_derivative(activation):
    """Takes sigmoid(z) and returns d sigmoid(z) / dz, so the forward pass can be reused."""
    return activation * (1 - activation)


def softmax(x):
    if x.ndim == 1:
        exps = np.exp(x - np.max(x))
        return exps / np.sum(exps)
    elif x.ndim == 2:
        exps = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exps / np.sum(exps, axis=1, keepdims=True)


def conv2d(image, kernel):
    """Valid cross-correlation, one window at a time."""
    h, w = image.shape
    kh, kw = kernel.shape
    output = np.zeros((h - kh + 1, w - kw + 1))
    for i in range(output.shape[0]):
        for j in range(output.shape[1]):
            output[i, j] = np.sum(image[i:i + kh, j:j + kw] * kernel)
    return output


def avg_pooling(feature_map, pool_size, stride):
    h, w = feature_map.shape
    ph, pw = pool_size
    output = []
    for i in range(0, h - ph + 1, stride):
        row = []
        for j in range(0, w - pw + 1, stride):
            row.append(np.mean(feature_map[i:i + ph, j:j + pw]))
        output.append(row)
    return np.array(output)


def cross_entropy_loss(y_pred, y_true):
    return -np.sum(y_true * np.log(y_pred + 1e-15))  # 1e-15 keeps log away from zero

In [ ]:
def initialize_weights():
    np.random.seed(0)  # fixed seed, so the numbers below reproduce exactly
    conv_kernel = np.random.randn(2, 3, 3) * 0.1
    fc_weights = np.random.randn(10, 2 * 13 * 13) * 0.1
    return conv_kernel, fc_weights


def forward_propagation(x, conv_kernel, fc_weights):
    conv_outputs = [sigmoid(conv2d(x, kernel)) for kernel in conv_kernel]
    pooled_outputs = [avg_pooling(output, (2, 2), 2) for output in conv_outputs]
    flattened = np.concatenate([output.flatten() for output in pooled_outputs])
    logits = fc_weights @ flattened
    predictions = softmax(logits)
    return predictions, conv_outputs, pooled_outputs, flattened, logits


def backward_propagation(x, y_true, conv_outputs, pooled_outputs, flattened, logits,
                         conv_kernel, fc_weights, lr=0.01):
    # softmax and cross-entropy collapse into a single term
    d_logits = softmax(logits) - y_true
    d_fc_weights = np.outer(d_logits, flattened)
    d_flattened = fc_weights.T @ d_logits

    kernel_grads = np.zeros_like(conv_kernel)
    offset = 0
    for i, pooled in enumerate(pooled_outputs):
        d_pooled = d_flattened[offset:offset + pooled.size].reshape(pooled.shape)
        offset += pooled.size

        # each pooled cell is the mean of a 2x2 block, so every cell in that
        # block gets a quarter of the pooled gradient back
        d_activation = np.repeat(np.repeat(d_pooled, 2, axis=0), 2, axis=1) / 4
        d_pre_activation = d_activation * sigmoid_derivative(conv_outputs[i])
        kernel_grads[i] = conv2d(x, d_pre_activation)

    fc_weights -= lr * d_fc_weights
    conv_kernel -= lr * kernel_grads


def train(x_train, y_train, conv_kernel, fc_weights, epochs=10, lr=0.01):
    for epoch in range(epochs):
        loss = 0
        for x, y in zip(x_train, y_train):
            y_true = one_hot_encode(y)
            predictions, conv_outputs, pooled_outputs, flattened, logits = forward_propagation(
                x, conv_kernel, fc_weights)
            loss += cross_entropy_loss(predictions, y_true)
            backward_propagation(x, y_true, conv_outputs, pooled_outputs, flattened, logits,
                                 conv_kernel, fc_weights, lr)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss / len(x_train)}")


def evaluate(x_test, y_test, conv_kernel, fc_weights):
    correct = 0
    for x, y in zip(x_test, y_test):
        predictions, *_ = forward_propagation(x, conv_kernel, fc_weights)
        if np.argmax(predictions) == y:
            correct += 1
    accuracy = correct / len(x_test)
    print(f"Accuracy: {accuracy * 100:.2f}%")

In [ ]:
x_train, y_train, x_test, y_test = load_data()
conv_kernel, fc_weights = initialize_weights()

train(x_train, y_train, conv_kernel, fc_weights, epochs=10, lr=0.01)
evaluate(x_test, y_test, conv_kernel, fc_weights)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10, Loss: 0.05028540449492675
Epoch 2/10, Loss: 0.038176612262655424
Epoch 3/10, Loss: 0.03637649503355718
Epoch 4/10, Loss: 0.03547008456537964
Epoch 5/10, Loss: 0.03490383922425014
Epoch 6/10, Loss: 0.034509492658570236
Epoch 7/10, Loss: 0.03421602293758765
Epoch 8/10, Loss: 0.0339876292365444
Epoch 9/10, Loss: 0.033804064069005886
Epoch 10/10, Loss: 0.03365290203624081
Accuracy: 88.19%
